<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left">  Exercise: Fix a Broken Agent Solution_ver</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise: Fix a Broken Agent

**Purpose:**  
Practice debugging and fixing a faulty AI agent by analyzing traces and modifying prompts/tools.

**Realistic Scenario:**  
You are building a Travel Planning Agent for users visiting Riyadh, Saudi Arabia
.  
The agent should:
- Get current weather in Riyadh
- Suggest 2-3 popular places to visit
- Calculate approximate total cost for a 2-day trip (hotel + food + transport)

However, the current agent is broken and shows common failure patterns.

**Your Task:**
1. Run the broken agent and carefully study the traces
2. Identify the problems (wrong tool, looping, premature answer, etc.)
3. Fix the agent by modifying the prompt, tool descriptions, or stopping conditions
4. Compare the traces before and after your fixes

In [ ]:
# Setup - Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/15

#Tools

In [ ]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    if "Riyadh" in city.lower():
        return "Weather in Riyadh: 28°C, Sunny with light wind."
    return f"Weather in {city}: 25°C, Sunny (dummy data)"

@tool
def calculate_math(expression: str) -> str:
    """Perform calculations. Example: '2500 * 2 + 1500'"""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except:
        return "Calculation error"

@tool
def scrape_webpage(url: str) -> str:
    """Scrape tourism information from a webpage."""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.title.string.strip() if soup.title else "No title"
        return f"Title: {title}\nContent: Tourism information for Riyadh area."
    except Exception as e:
        return f"Scraping failed: {str(e)}"

tools = [get_weather, calculate_math, scrape_webpage]
print("Tools ready")

Tools ready


## Broken Agent - fixed

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# Fixed and improved prompt
fixed_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

You are a helpful travel planning assistant for riyadh.

Good reasoning flow:
1. First get the weather using get_weather tool
2. Then get tourism information using scrape_webpage if needed
3. Finally use calculate_math to compute total estimated cost

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (clean string only)
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

fixed_prompt = PromptTemplate.from_template(fixed_prompt_template)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

fixed_agent = create_react_agent(llm=llm, tools=tools, prompt=fixed_prompt)

fixed_executor = AgentExecutor(
    agent=fixed_agent,
    tools=tools,
    verbose=True,
    max_iterations=12,
    handle_parsing_errors=True
)

print(" Fixed Travel Planning Agent")

response = fixed_executor.invoke({
    "input": "Plan a 2-day trip to riyadh. Include weather, top places to visit, and total estimated cost."
})

print("\nFinal Output:", response["output"])

 Fixed Travel Planning Agent


> Entering new AgentExecutor chain...
Thought: you should always think about what to do
Action: get_weather
Action Input: riyadhWeather in riyadh: 25°C, Sunny (dummy data)Now that we know the weather, let's find out the top places to visit in Riyadh.
Action: scrape_webpage
Action Input: top places to visit in RiyadhScraping failed: Invalid URL 'top places to visit in Riyadh': No scheme supplied. Perhaps you meant https://top places to visit in Riyadh?We need to provide a valid URL to scrape tourism information. Let's try again.
Action: scrape_webpage
Action Input: https://www.tripadvisor.com/Attractions-g293995-Activities-Riyadh_Riyadh_Province.htmlTitle: tripadvisor.com
Content: Tourism information for Riyadh area.Now that we have tourism information, let's calculate the total estimated cost for the trip.
Action: calculate_math
Action Input: 1500 + 500 + 1000 (dummy cost values)Calculation errorThere seems to be an error in the calculation input. Let's t

## Instructor Solution Explanation

The original broken agent had these issues:
- Wrong tool selection (forced to always start with get_weather)
- Poor Action Input formatting
- No clear reasoning steps for multi-tool workflow

**Main fixes applied:**
- Removed the bad "CRITICAL RULE"
- Added clear step-by-step reasoning guidance
- Added instruction for clean Action Input
- Increased max_iterations slightly

After fixing, the trace shows proper multi-step reasoning:
1. Get weather → 2. Get tourism info → 3. Calculate costs